# 🧠 Personal Memory Vector Database: Ingestion & Exploration

In this notebook, we take **`personal_knowledge_dump_vector_db.txt`**—your comprehensive personal export—and convert it into a searchable **Vector Memory Brain** using ChromaDB!

### 🎯 Learning Goals:
1. **Load `personal_knowledge_dump_vector_db.txt`** cleanly.
2. **Implement Heading-Aware Chunking**: Preserving the 14 major sections and 49 subsections (Hardware, KineticFit, Academics, Fitness, Automobiles, etc.).
3. **Store in ChromaDB**: Embed and persist in `./my_local_chroma_db` under collection `personal_knowledge_base`.
4. **Run Semantic Recall Tests**: Test natural language questions about your laptop, projects, and goals.
5. **Visual Exploration**: Connect to the live Web UI at **`http://localhost:5050`**.

---
## Step 1: Read the Personal Knowledge Dump File

Let's inspect the raw text file located in our workspace:

In [ ]:
import os

file_path = os.path.join(os.getcwd(), "personal_knowledge_dump_vector_db.txt")

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f" Loaded file: {file_path}")
print(f"📏 File Size: {len(raw_text):,} characters / {len(raw_text.splitlines())} lines")
print("\n--- First 300 Characters ---")
print(raw_text[:300])

---
## Step 2: Heading-Aware Chunking Strategy

Because this file contains structured headers (`======================================================= N. TITLE`) and sub-points (`1.1`, `2.2`, etc.), we parse each subsection as an independent, coherent chunk.

Each chunk gets tagged with:
- **`section`**: The main category title (e.g., *Computers, Hardware, Operating Systems*)
- **`topic`**: The specific subsection (e.g., *2.2 Current laptop: HP OMEN 16-am0240TX*)
- **`category`**: A high-level tag (e.g., *Hardware & Computers*, *Startups & Projects*, *Fitness & Running*)

In [ ]:
import re

# Split by major section dividers
parts = re.split(r'={20,}\s*(\d+\..*?)\s*={20,}', raw_text, flags=re.DOTALL)

chunks = []
metadatas = []
ids = []
counter = 1

category_map = {
    "1": "Profile & Academics",
    "2": "Hardware & Computers",
    "3": "Startups & Projects",
    "4": "AI & Data Systems",
    "5": "Fitness & Running",
    "6": "Personal Health",
    "7": "Driving Licence",
    "8": "Automobiles",
    "9": "Interviews & Codebases",
    "10": "Branding & Naming",
    "11": "Preferences & Style",
    "12": "Vector DB Testing Guide",
    "13": "Chronology Snapshot",
    "14": "Limitations & Scope"
}

# Intro
if parts[0].strip():
    chunks.append(parts[0].strip())
    metadatas.append({"section": "0. Intro", "topic": "Purpose", "category": "Overview"})
    ids.append(f"pkd_{counter:03d}")
    counter += 1

# Sections 1 through 14
for i in range(1, len(parts), 2):
    sec_header = re.sub(r'\s+', ' ', parts[i]).strip()
    sec_content = parts[i+1].strip()
    sec_num = sec_header.split('.')[0].strip()
    cat = category_map.get(sec_num, "General")
    
    sub_matches = list(re.finditer(r'(?:^|\n\n)(\d+\.\d+)\s+([^\n\-]+?)\s*-\s*', sec_content))
    if sub_matches:
        for idx, sm in enumerate(sub_matches):
            start = sm.start()
            end = sub_matches[idx+1].start() if idx + 1 < len(sub_matches) else len(sec_content)
            sub_num = sm.group(1)
            sub_title = sm.group(2).strip()
            chunk_body = sec_content[start:end].strip()
            
            chunk_text = f"[{sec_header} | {sub_num} {sub_title}]\n{chunk_body}"
            chunks.append(chunk_text)
            metadatas.append({"section": sec_header, "topic": f"{sub_num} {sub_title}", "category": cat})
            ids.append(f"pkd_{counter:03d}")
            counter += 1
    elif sec_num == "13":
        entries = re.split(r'\n\n(?=[A-Za-z0-9])', sec_content)
        for e in entries:
            if e.strip():
                first_line = e.strip().split('\n')[0].strip()
                chunk_text = f"[{sec_header} | Timeline: {first_line}]\n{e.strip()}"
                chunks.append(chunk_text)
                metadatas.append({"section": sec_header, "topic": first_line, "category": cat})
                ids.append(f"pkd_{counter:03d}")
                counter += 1
    else:
        chunks.append(f"[{sec_header}]\n{sec_content}")
        metadatas.append({"section": sec_header, "topic": sec_header, "category": cat})
        ids.append(f"pkd_{counter:03d}")
        counter += 1

print(f" Generated {len(chunks)} structured memory chunks!")

---
## Step 3: Persist Chunks into ChromaDB

We now connect to our Persistent Client in `./my_local_chroma_db` and upsert all 49 chunks into collection **`personal_knowledge_base`**.

In [ ]:
import chromadb

db_dir = os.path.join(os.getcwd(), "my_local_chroma_db")
client = chromadb.PersistentClient(path=db_dir)

collection = client.get_or_create_collection(
    name="personal_knowledge_base",
    metadata={"description": "User personal knowledge dump chunks"}
)

collection.upsert(ids=ids, documents=chunks, metadatas=metadatas)

print(f" Successfully indexed {collection.count()} chunks in collection '{collection.name}'!")
print(f"📁 Persistent storage location: {db_dir}")

---
## Step 4: Test Semantic Recall on Personal Questions

Let's test asking natural language questions about your hardware, projects, running goals, and academic history:

In [ ]:
test_queries = [
    "What laptop do I currently use and what are its specs?",
    "Tell me about KineticFit tech stack and vision",
    "What is my reported 10 km running time?",
    "What were the ground clearance comparisons for sedans versus SUVs?",
    "What is my current reported BCA CGPA?"
]

for q in test_queries:
    res = collection.query(query_texts=[q], n_results=1)
    doc_id = res["ids"][0][0]
    meta = res["metadatas"][0][0]
    dist = res["distances"][0][0]
    text = res["documents"][0][0]
    
    print(f"🔍 Question: '{q}'")
    print(f"   🎯 Top Match: {meta['category']} -> {meta['topic']} (Distance: {dist:.4f})")
    print(f"   📄 Excerpt:\n{text[:220]}...\n")
    print("-" * 70)

---
## Step 5: Explore Visually in Your Web Browser!

You can now search and browse all 49 chunks interactively in the web browser at:
👉 **[http://localhost:5050](http://localhost:5050)**

- View cards categorized by Hardware, Projects, Fitness, Academics, Health, and Automobiles.
- Inspect raw 384-dimensional vector coordinates.
- Run live semantic searches with match percentages!